# Brazo robótico · Grupo 3
**Mateo Flores y Federico Soler — Introducción a la Robótica, Universidad de Palermo**

Este notebook representa un brazo con **giro de base y elevación de hombro** a partir de las consignas angulares registradas por el programa de Arduino en Wokwi.

**Para verlo:** elegir **Entorno de ejecución → Ejecutar todas**. La captura real ya está incluida: no hace falta subir archivos para la primera reproducción. El gráfico de ángulos, el brazo y el reloj avanzan juntos. El reproductor permite pausar y recorrer los cuadros.

**Flujo:** programa de Arduino en Wokwi → monitor serie → CSV con tiempos y ángulos → cinemática directa → brazo animado.

La captura incluida contiene **271 registros y 5,488 s** de ejecución del sketch instrumentado. Se obtuvo en una sesión temporal del [proyecto original de Wokwi](https://wokwi.com/projects/477503605988146177), sin guardar cambios en el proyecto publicado. Los datos son las órdenes enviadas a los servos, no mediciones de su movimiento real.

Repositorio del proyecto: https://github.com/federicosolerup/brazo-robotico-arduino-wokwi


## 1. Modelo del brazo

Se representan **dos grados de libertad**: base alrededor del eje Z y hombro que eleva un único eslabón móvil. No se agrega un codo móvil.

Las dimensiones son ilustrativas: altura del hombro $h=8\,\mathrm{cm}$, desplazamiento fijo respecto del eje de la base $d=3\,\mathrm{cm}$ y largo del brazo $L=18\,\mathrm{cm}$. No corresponden a un prototipo físico medido.

Se toma X hacia el frente, Y hacia la izquierda y Z hacia arriba. La correspondencia con los servos es:

$$\varphi=90^\circ-q_{base},\qquad \beta=q_{hombro}.$$

Así, base = 90° mira hacia +X y hombro = 90° deja el eslabón vertical. La posición del extremo se calcula con cinemática directa:

$$x=(d+L\cos\beta)\cos\varphi,\quad y=(d+L\cos\beta)\sin\varphi,\quad z=h+L\sin\beta.$$

La separación $d$ es fija; permite visualizar el giro de la base incluso con el eslabón vertical. La secuencia es:

| Configuración | Base | Hombro |
|---|---:|---:|
| P0 · reposo | 90° | 90° |
| P1 | 30° | 45° |
| P2 | 150° | 45° |
| P3 · regreso | 90° | 90° |

Cada tramo usa el perfil cúbico $q(t)=q_0+(q_f-q_0)(3s^2-2s^3)$, con $s=t/T$ y $T=1500\,\mathrm{ms}$. Su velocidad ideal es nula al comenzar y terminar. La aceleración no tiene por qué ser nula. Arduino envía ángulos enteros, por lo que las consignas tienen cuantización.

## 2. Funciones de lectura, geometría y animación

Esta celda contiene todo el código necesario. Usa NumPy y Matplotlib, disponibles en Colab. El mismo código se entrega como `visualizar_brazo.py` para ejecución local.

In [ ]:
# @title Código de lectura y visualización (expandir para inspeccionar)
"""Visualizacion cinematica de un brazo con giro de base y elevacion de hombro.

Uso local: python visualizar_brazo.py --csv datos/captura_wokwi.csv
Sin --csv se usan datos teoricos, identificados como demostracion.
El notebook incluye estas mismas funciones y funciona sin descargar este archivo.
"""
from pathlib import Path
import argparse
import csv
import io
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

PUNTOS = np.array([[90, 90], [30, 45], [150, 45], [90, 90]])
ALTURA_CM = 8.0
DESPLAZAMIENTO_CM = 3.0
LARGO_CM = 18.0
AZUL = '#176B9B'
NARANJA = '#DC7137'
TINTA = '#23384A'
FONDO = '#F7F8FA'


def datos_demostracion():
    """Calcula un ejemplo teorico; NO es una captura del simulador."""
    filas = []
    reloj = 0
    for i, (inicio, fin) in enumerate(zip(PUNTOS[:-1], PUNTOS[1:]), 1):
        for ms in range(0, 1501, 20):
            s = ms / 1500
            # Aproxima la conversion de Servo.write a grados enteros.
            q = (inicio + (fin - inicio) * (3*s*s - 2*s*s*s)).astype(int)
            filas.append([1, reloj + ms, *q, f'tramo_{i}'])
        reloj += 1520  # El original hace delay(20) tambien al llegar al extremo.
        for ms in range(0, 301, 20):
            filas.append([1, reloj + ms, *fin, f'pausa_{i}'])
        reloj += 300
    filas.append([1, reloj, 90, 90, 'fin'])
    texto = io.StringIO()
    w = csv.writer(texto, lineterminator='\n')
    w.writerow(['ciclo', 'tiempo_ms', 'base_deg', 'hombro_deg', 'fase'])
    w.writerows(filas)
    return cargar_csv(texto.getvalue(), origen='DEMOSTRACION TEORICA · sin captura de Wokwi')


def cargar_csv(texto, ciclo=None, origen='CSV importado · consignas angulares'):
    """Lee la salida serial. Conserva la ultima sesion y el ultimo ciclo elegible.

    Acepta comentarios #, encabezados repetidos y tiempos duplicados. Ante un
    reinicio del simulador, un nuevo encabezado separa las sesiones. Nunca ordena
    tiempos descendentes para ocultar un registro mezclado o corrupto.
    """
    ansi = re.compile(r'\x1b\[[0-?]*[ -/]*[@-~]')
    registros = {}
    encabezado = None
    sesion = -1
    requeridas = {'tiempo_ms', 'base_deg', 'hombro_deg'}
    for numero, linea in enumerate(texto.splitlines(), 1):
        linea = ansi.sub('', linea).strip().lstrip('\ufeff')
        if not linea or linea.startswith('#'):
            continue
        campos = next(csv.reader([linea]))
        campos = [v.strip() for v in campos]
        if requeridas.issubset(campos):
            encabezado = campos
            sesion += 1
            continue
        if encabezado is None:
            # Permite copiar mensajes de arranque anteriores al encabezado.
            continue
        if len(campos) != len(encabezado):
            raise ValueError(f'Linea {numero}: faltan columnas o la fila esta incompleta.')
        fila = dict(zip(encabezado, campos))
        try:
            c = int(fila.get('ciclo', '1'))
            t, b, h = [float(fila[k]) for k in ['tiempo_ms', 'base_deg', 'hombro_deg']]
        except ValueError as exc:
            raise ValueError(f'Linea {numero}: tiempo o angulos no numericos.') from exc
        if not np.isfinite([t, b, h]).all() or t < 0 or not (0 <= b <= 180 and 0 <= h <= 180):
            raise ValueError(f'Linea {numero}: tiempo o angulos fuera de rango.')
        registros.setdefault((sesion, c), []).append((t, b, h, fila.get('fase', 'registro')))
    elegibles = [k for k in registros if ciclo is None or k[1] == ciclo]
    if not elegibles:
        raise ValueError('No hay registros validos. Copiar el encabezado y un ciclo completo.')
    clave = elegibles[-1]
    filas = registros[clave]
    if any(b[0] < a[0] for a, b in zip(filas, filas[1:])):
        raise ValueError('El tiempo retrocede dentro de un ciclo. Revisar la captura.')
    # Varias consignas en el mismo ms: queda vigente la ultima de ese instante.
    unicas = {f[0]: f for f in filas}
    filas = list(unicas.values())
    if len(filas) < 2:
        raise ValueError('Se necesitan al menos dos instantes distintos.')
    valores = np.array([f[:3] for f in filas], dtype=float)
    return {
        't': (valores[:, 0] - valores[0, 0]) / 1000,
        'base': valores[:, 1], 'hombro': valores[:, 2],
        'fase': np.array([f[3] for f in filas]),
        'origen': origen, 'ciclo': clave[1], 'sesion': clave[0] + 1,
    }


def cinematica(base, hombro, altura=ALTURA_CM, offset=DESPLAZAMIENTO_CM, largo=LARGO_CM):
    """Modelo Rz(phi) · Tx(offset) · Tz(altura) · Ry(-beta) · Tx(largo).

    phi = 90 - base (90 grados es el frente +X); beta = hombro, medido
    desde la horizontal. +Y corresponde al lado izquierdo del robot.
    Las longitudes son ilustrativas y el offset es un eslabon fijo.
    """
    phi, beta = np.deg2rad(90 - np.asarray(base)), np.deg2rad(hombro)
    hombro_xyz = np.stack([offset*np.cos(phi), offset*np.sin(phi),
                          np.broadcast_to(altura, np.shape(phi))], axis=-1)
    extremo = hombro_xyz + np.stack([largo*np.cos(beta)*np.cos(phi),
                                    largo*np.cos(beta)*np.sin(phi),
                                    largo*np.sin(beta)], axis=-1)
    return hombro_xyz, extremo


def informe(datos):
    """Verifica las consignas registradas contra la secuencia del TP."""
    q = np.column_stack([datos['base'], datos['hombro']])
    encontrados = []
    cursor = 0
    for punto in PUNTOS:
        indices = np.flatnonzero(np.all(np.isclose(q[cursor:], punto), axis=1))
        if not len(indices):
            break
        cursor += int(indices[0])
        encontrados.append(float(datos['t'][cursor]))
        cursor += 1
    segundo = datos['fase'] == 'tramo_2'
    fijo = bool(np.allclose(datos['hombro'][segundo], 45)) if segundo.any() else None
    return {
        'origen': datos['origen'], 'ciclo': datos['ciclo'],
        'muestras': len(q), 'duracion_s': round(float(datos['t'][-1]), 3),
        'configuraciones_en_orden': len(encontrados) == 4,
        'primera_consigna_objetivo_s': encontrados,
        'hombro_fijo_en_tramo_2': fijo,
        'regresa_a_90_90': bool(np.allclose(q[-1], [90, 90])),
    }


def crear_animacion(datos, fps=25):
    """Devuelve figura y animacion. Mantiene cada consigna hasta la siguiente."""
    if not 1 <= fps <= 60:
        raise ValueError('Elegir una frecuencia entre 1 y 60 cuadros por segundo.')
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                         'text.color': TINTA, 'axes.labelcolor': TINTA,
                         'xtick.color': TINTA, 'ytick.color': TINTA})
    fig = plt.figure(figsize=(13.6, 7.7), facecolor=FONDO)
    fig.text(.045, .935, 'Brazo robótico', fontsize=25, weight='bold')
    fig.text(.045, .895, 'GRUPO 3   /   Mateo Flores · Federico Soler', fontsize=10, color='#617486')
    fig.text(.955, .933, '2 GRADOS DE LIBERTAD', ha='right', fontsize=11, weight='bold', color=AZUL)
    fig.text(.955, .895, datos['origen'], ha='right', fontsize=9, color='#617486')
    ax = fig.add_axes([.025, .17, .59, .69], projection='3d', facecolor=FONDO)
    ax.view_init(elev=22, azim=40)
    ax.set(xlim=(-7, 24), ylim=(-21, 21), zlim=(0, 29),
           xlabel='X · frente (cm)', ylabel='Y · izquierda (cm)', zlabel='Z (cm)')
    ax.set_box_aspect((31, 42, 29))
    ax.set_xticks([0, 10, 20]); ax.set_yticks([-20, 0, 20]); ax.set_zticks([0, 10, 20])
    ax.tick_params(labelsize=8, pad=0)
    for eje in [ax.xaxis, ax.yaxis, ax.zaxis]:
        eje.pane.fill = False
        eje.pane.set_edgecolor('#DAE1E7')
        eje._axinfo['grid']['color'] = '#E0E6EC'
    a = np.linspace(0, 2*np.pi, 40)
    z = np.array([0, ALTURA_CM - .8])
    xx = 3.4 * np.cos(a)[None, :] * np.ones((2, 1))
    yy = 3.4 * np.sin(a)[None, :] * np.ones((2, 1))
    zz = np.broadcast_to(z[:, None], xx.shape)
    ax.plot_surface(xx, yy, zz, color='#9AAAB6', alpha=.85, linewidth=0, shade=True)
    ax.plot(4.4*np.cos(a), 4.4*np.sin(a), np.full_like(a, ALTURA_CM),
            color='#ABB9C5', ls='--', lw=1)
    hombros, extremos = cinematica(datos['base'], datos['hombro'])
    _, objetivos = cinematica(PUNTOS[:, 0], PUNTOS[:, 1])
    for i, punto in enumerate(objetivos[:3]):
        ax.scatter(*punto, s=24, color='#A7B5C0', depthshade=False)
        ax.text(punto[0], punto[1], punto[2]+1.1, ['P0 / P3', 'P1', 'P2'][i],
                fontsize=9, color='#708596')
    ax.plot(extremos[:, 0], extremos[:, 1], extremos[:, 2], color='#D6E0E7', lw=1.2)
    (rastro,) = ax.plot([], [], [], color=AZUL, alpha=.65, lw=2)
    (soporte,) = ax.plot([], [], [], color=NARANJA, lw=7, solid_capstyle='round')
    (brazo,) = ax.plot([], [], [], color=AZUL, lw=13, solid_capstyle='round')
    (juntas,) = ax.plot([], [], [], 'o', color=NARANJA, ms=10, mec='white', mew=2)
    (punta,) = ax.plot([], [], [], 'o', color=AZUL, ms=8, mec='white', mew=2)
    (proyeccion,) = ax.plot([], [], [], color='#B5C6D1', ls=':', lw=1)
    plataforma = Poly3DCollection([], facecolors=NARANJA, edgecolors='white', alpha=.8)
    ax.add_collection3d(plataforma)

    graf = fig.add_axes([.67, .39, .29, .33], facecolor='white')
    graf.plot(datos['t'], datos['base'], color=AZUL, lw=2, drawstyle='steps-post', label='Base · D9')
    graf.plot(datos['t'], datos['hombro'], color=NARANJA, lw=2, drawstyle='steps-post', label='Hombro · D10')
    graf.set(xlim=(0, max(float(datos['t'][-1]), .1)), ylim=(20, 165), xlabel='Tiempo (s)', ylabel='Consigna (°)')
    graf.set_yticks([30, 45, 90, 150]); graf.grid(alpha=.15)
    graf.spines[['top', 'right']].set_visible(False)
    graf.legend(frameon=False, loc='upper left', fontsize=9)
    cursor = graf.axvline(0, color=TINTA, lw=1.1, ls='--')
    (punto_b,) = graf.plot([], [], 'o', color=AZUL, ms=6)
    (punto_h,) = graf.plot([], [], 'o', color=NARANJA, ms=6)
    estado = fig.text(.67, .81, '', fontsize=15, weight='bold')
    reloj = fig.text(.67, .765, '', fontsize=11, color='#617486')
    angulos = fig.text(.67, .295, '', fontsize=16, weight='bold')
    posicion = fig.text(.67, .25, '', fontsize=10, color='#617486')
    fig.text(.045, .105, 'MODELO CINEMÁTICO', fontsize=9, weight='bold', color=AZUL)
    fig.text(.045, .074, 'Base giratoria + hombro + un eslabón móvil. Dimensiones ilustrativas: h = 8 cm, d = 3 cm, L = 18 cm.', fontsize=9)
    fig.text(.045, .044, 'Se representan consignas de posición; el modelo no calcula cargas, torque ni error de seguimiento.', fontsize=9, color='#617486')
    nombres = {'inicio': 'P0 · Reposo', 'tramo_1': '01 · Hacia P1', 'pausa_1': 'P1 · Pausa',
               'tramo_2': '02 · Giro con hombro fijo', 'pausa_2': 'P2 · Pausa',
               'tramo_3': '03 · Regreso a P0', 'pausa_3': 'P3 · Reposo final', 'fin': 'Secuencia completa'}
    instantes = np.arange(0, datos['t'][-1], 1/fps)
    instantes = np.r_[instantes, datos['t'][-1]]

    def actualizar(t):
        i = min(int(np.searchsorted(datos['t'], t, side='right')-1), len(datos['t'])-1)
        i = max(i, 0)
        s, e = hombros[i], extremos[i]
        phi = np.deg2rad(90-datos['base'][i])
        u = np.array([np.cos(phi), np.sin(phi), 0.])
        v = np.array([-np.sin(phi), np.cos(phi), 0.])
        centro = np.array([0., 0., ALTURA_CM])
        plataforma.set_verts([[centro+du*u+dv*v for du, dv in [(-2,-1.5),(4,-1.5),(4,1.5),(-2,1.5)]]])
        soporte.set_data_3d([0, s[0]], [0, s[1]], [ALTURA_CM, s[2]])
        brazo.set_data_3d([s[0], e[0]], [s[1], e[1]], [s[2], e[2]])
        juntas.set_data_3d([s[0]], [s[1]], [s[2]])
        punta.set_data_3d([e[0]], [e[1]], [e[2]])
        rastro.set_data_3d(extremos[:i+1, 0], extremos[:i+1, 1], extremos[:i+1, 2])
        proyeccion.set_data_3d([e[0], e[0]], [e[1], e[1]], [0, e[2]])
        cursor.set_xdata([t, t])
        punto_b.set_data([t], [datos['base'][i]])
        punto_h.set_data([t], [datos['hombro'][i]])
        estado.set_text(nombres.get(datos['fase'][i], 'Reproducción del registro'))
        reloj.set_text(f't = {t:0.2f} s   /   {datos["t"][-1]:0.2f} s')
        angulos.set_text(f'Base {datos["base"][i]:.0f}°    Hombro {datos["hombro"][i]:.0f}°')
        posicion.set_text(f'Extremo (cm):  x {e[0]:.1f}   y {e[1]:.1f}   z {e[2]:.1f}')
        return brazo, soporte, juntas, punta, rastro, cursor, estado

    actualizar(0)
    anim = FuncAnimation(fig, actualizar, frames=instantes, interval=1000/fps, blit=False, repeat=False)
    # Permite guardar capturas de instantes concretos sin recalcular otra escena.
    anim.actualizar_instante = actualizar
    return fig, anim


def guardar_html(anim, destino, fps=25):
    plt.rcParams['animation.embed_limit'] = 80
    with plt.rc_context({'animation.frame_format': 'jpeg', 'savefig.dpi': 90}):
        reproductor = anim.to_jshtml(fps=fps, default_mode='once')
    # Los cuadros y el JS quedan embebidos. Sustituye los iconos de un CDN
    # por caracteres locales para que el reproductor funcione sin internet.
    reproductor = re.sub(r'<link rel="stylesheet"[^>]*font-awesome[^>]*>', '', reproductor)
    reproductor = reproductor.replace('<i class="fa fa-play fa-flip-horizontal"></i>', '◀')
    iconos = {'fa-fast-backward': '⏮', 'fa-step-backward': '◀', 'fa-backward': '◁',
              'fa-pause': 'Ⅱ', 'fa-play': '▶', 'fa-forward': '▷',
              'fa-step-forward': '▶', 'fa-fast-forward': '⏭',
              'fa-minus': '−', 'fa-plus': '+'}
    for clase, simbolo in iconos.items():
        reproductor = reproductor.replace(f'<i class="fa {clase}"></i>', simbolo)
    pagina = ('<!doctype html><html lang="es"><meta charset="utf-8">'
              '<meta name="viewport" content="width=device-width, initial-scale=1">'
              '<title>Brazo robótico · Grupo 3</title>'
              '<style>body{margin:0;background:#F7F8FA;font-family:Arial,sans-serif}'
              '.animation{margin:0 auto;max-width:1360px}.animation img{max-width:100%;height:auto}'
              'button,input,select{font-size:15px}</style>' + reproductor + '</html>')
    Path(destino).write_text(pagina, encoding='utf-8')

## 3. Elegir los datos

- **`captura_incluida`**: reproduce el CSV real capturado en Wokwi; es el valor inicial.
- **`subir_csv`**: solicita subir una nueva captura del monitor serie desde el equipo.
- **`pegar_csv`**: lee el texto pegado en `CSV_PEGADO`, incluido su encabezado.
- **`demostracion`**: calcula datos teóricos para explorar el modelo. Se identifica expresamente como demostración.

Si hay varios ciclos, se elige el último. Para seleccionar uno, reemplazar `CICLO = None` por su número. Si hubo reinicios, se toma la última sesión disponible. El lector rechaza valores inválidos y tiempos que retroceden dentro de un mismo ciclo.

In [ ]:
# @title Fuente de datos
MODO = "captura_incluida"  # @param ["captura_incluida", "subir_csv", "pegar_csv", "demostracion"]
CICLO = None  # None = último ciclo; también se puede escribir 1, 2, etc.
FPS = 20

CSV_PEGADO = """
"""

In [ ]:
# @title Captura de Wokwi incluida y su procedencia
CSV_INCLUIDO = 'ciclo,tiempo_ms,base_deg,hombro_deg,fase\n1,0,90,90,inicio\n1,1,90,90,tramo_1\n1,22,89,89,tramo_1\n1,42,89,89,tramo_1\n1,63,89,89,tramo_1\n1,84,89,89,tramo_1\n1,105,89,89,tramo_1\n1,125,88,89,tramo_1\n1,147,88,88,tramo_1\n1,167,87,88,tramo_1\n1,187,87,88,tramo_1\n1,208,86,87,tramo_1\n1,229,86,87,tramo_1\n1,250,85,86,tramo_1\n1,270,84,86,tramo_1\n1,292,84,85,tramo_1\n1,312,83,84,tramo_1\n1,333,82,84,tramo_1\n1,354,81,83,tramo_1\n1,375,80,82,tramo_1\n1,395,79,82,tramo_1\n1,417,78,81,tramo_1\n1,437,77,80,tramo_1\n1,458,76,79,tramo_1\n1,478,75,79,tramo_1\n1,500,74,78,tramo_1\n1,520,73,77,tramo_1\n1,541,72,76,tramo_1\n1,562,71,75,tramo_1\n1,583,69,74,tramo_1\n1,603,68,74,tramo_1\n1,625,67,73,tramo_1\n1,645,66,72,tramo_1\n1,666,65,71,tramo_1\n1,687,63,70,tramo_1\n1,708,62,69,tramo_1\n1,728,61,68,tramo_1\n1,749,60,67,tramo_1\n1,770,58,66,tramo_1\n1,791,57,65,tramo_1\n1,811,56,64,tramo_1\n1,833,55,63,tramo_1\n1,853,53,62,tramo_1\n1,874,52,61,tramo_1\n1,895,51,61,tramo_1\n1,916,50,60,tramo_1\n1,936,49,59,tramo_1\n1,956,47,58,tramo_1\n1,978,46,57,tramo_1\n1,998,45,56,tramo_1\n1,1019,44,55,tramo_1\n1,1040,43,55,tramo_1\n1,1061,42,54,tramo_1\n1,1081,41,53,tramo_1\n1,1103,40,52,tramo_1\n1,1123,39,52,tramo_1\n1,1144,38,51,tramo_1\n1,1165,37,50,tramo_1\n1,1186,36,50,tramo_1\n1,1206,36,49,tramo_1\n1,1228,35,48,tramo_1\n1,1248,34,48,tramo_1\n1,1269,33,47,tramo_1\n1,1290,33,47,tramo_1\n1,1311,32,46,tramo_1\n1,1331,32,46,tramo_1\n1,1353,31,46,tramo_1\n1,1373,31,45,tramo_1\n1,1394,30,45,tramo_1\n1,1415,30,45,tramo_1\n1,1436,30,45,tramo_1\n1,1456,30,45,tramo_1\n1,1478,30,45,tramo_1\n1,1498,30,45,tramo_1\n1,1519,30,45,tramo_1\n1,1540,30,45,pausa_1\n1,1561,30,45,pausa_1\n1,1581,30,45,pausa_1\n1,1602,30,45,pausa_1\n1,1622,30,45,pausa_1\n1,1643,30,45,pausa_1\n1,1663,30,45,pausa_1\n1,1684,30,45,pausa_1\n1,1704,30,45,pausa_1\n1,1726,30,45,pausa_1\n1,1746,30,45,pausa_1\n1,1766,30,45,pausa_1\n1,1787,30,45,pausa_1\n1,1807,30,45,pausa_1\n1,1828,30,45,pausa_1\n1,1839,30,45,pausa_1\n1,1840,30,45,pausa_1\n1,1841,30,45,tramo_2\n1,1862,30,45,tramo_2\n1,1882,30,45,tramo_2\n1,1904,30,45,tramo_2\n1,1924,31,45,tramo_2\n1,1945,31,45,tramo_2\n1,1966,32,45,tramo_2\n1,1987,33,45,tramo_2\n1,2007,34,45,tramo_2\n1,2029,35,45,tramo_2\n1,2049,36,45,tramo_2\n1,2070,37,45,tramo_2\n1,2091,38,45,tramo_2\n1,2112,40,45,tramo_2\n1,2132,41,45,tramo_2\n1,2154,43,45,tramo_2\n1,2174,45,45,tramo_2\n1,2195,46,45,tramo_2\n1,2216,48,45,tramo_2\n1,2236,50,45,tramo_2\n1,2257,52,45,tramo_2\n1,2278,54,45,tramo_2\n1,2299,56,45,tramo_2\n1,2319,58,45,tramo_2\n1,2341,61,45,tramo_2\n1,2361,63,45,tramo_2\n1,2382,65,45,tramo_2\n1,2403,68,45,tramo_2\n1,2424,70,45,tramo_2\n1,2444,72,45,tramo_2\n1,2466,75,45,tramo_2\n1,2486,77,45,tramo_2\n1,2507,80,45,tramo_2\n1,2528,82,45,tramo_2\n1,2549,85,45,tramo_2\n1,2569,87,45,tramo_2\n1,2591,90,45,tramo_2\n1,2611,92,45,tramo_2\n1,2632,95,45,tramo_2\n1,2653,97,45,tramo_2\n1,2674,100,45,tramo_2\n1,2694,102,45,tramo_2\n1,2716,104,45,tramo_2\n1,2736,107,45,tramo_2\n1,2758,109,45,tramo_2\n1,2778,112,45,tramo_2\n1,2799,114,45,tramo_2\n1,2820,116,45,tramo_2\n1,2841,118,45,tramo_2\n1,2862,121,45,tramo_2\n1,2883,123,45,tramo_2\n1,2903,125,45,tramo_2\n1,2925,127,45,tramo_2\n1,2945,129,45,tramo_2\n1,2966,131,45,tramo_2\n1,2987,133,45,tramo_2\n1,3008,134,45,tramo_2\n1,3029,136,45,tramo_2\n1,3050,138,45,tramo_2\n1,3070,139,45,tramo_2\n1,3092,141,45,tramo_2\n1,3112,142,45,tramo_2\n1,3134,143,45,tramo_2\n1,3154,144,45,tramo_2\n1,3174,145,45,tramo_2\n1,3196,146,45,tramo_2\n1,3216,147,45,tramo_2\n1,3237,148,45,tramo_2\n1,3258,148,45,tramo_2\n1,3279,149,45,tramo_2\n1,3300,149,45,tramo_2\n1,3321,149,45,tramo_2\n1,3341,150,45,tramo_2\n1,3362,150,45,pausa_2\n1,3383,150,45,pausa_2\n1,3404,150,45,pausa_2\n1,3424,150,45,pausa_2\n1,3445,150,45,pausa_2\n1,3465,150,45,pausa_2\n1,3486,150,45,pausa_2\n1,3507,150,45,pausa_2\n1,3528,150,45,pausa_2\n1,3548,150,45,pausa_2\n1,3569,150,45,pausa_2\n1,3589,150,45,pausa_2\n1,3610,150,45,pausa_2\n1,3631,150,45,pausa_2\n1,3652,150,45,pausa_2\n1,3662,150,45,pausa_2\n1,3663,150,45,tramo_3\n1,3683,149,45,tramo_3\n1,3705,149,45,tramo_3\n1,3725,149,45,tramo_3\n1,3746,149,45,tramo_3\n1,3767,149,45,tramo_3\n1,3788,148,45,tramo_3\n1,3809,148,46,tramo_3\n1,3830,147,46,tramo_3\n1,3850,147,46,tramo_3\n1,3872,146,47,tramo_3\n1,3892,146,47,tramo_3\n1,3914,145,48,tramo_3\n1,3934,144,48,tramo_3\n1,3955,144,49,tramo_3\n1,3976,143,50,tramo_3\n1,3997,142,50,tramo_3\n1,4018,141,51,tramo_3\n1,4039,140,52,tramo_3\n1,4059,139,52,tramo_3\n1,4081,138,53,tramo_3\n1,4101,137,54,tramo_3\n1,4123,136,55,tramo_3\n1,4143,135,55,tramo_3\n1,4164,134,56,tramo_3\n1,4185,133,57,tramo_3\n1,4206,132,58,tramo_3\n1,4226,130,59,tramo_3\n1,4248,129,60,tramo_3\n1,4268,128,61,tramo_3\n1,4290,127,61,tramo_3\n1,4310,126,62,tramo_3\n1,4331,124,63,tramo_3\n1,4352,123,64,tramo_3\n1,4373,122,65,tramo_3\n1,4394,121,66,tramo_3\n1,4415,119,67,tramo_3\n1,4435,118,68,tramo_3\n1,4457,117,69,tramo_3\n1,4477,116,70,tramo_3\n1,4499,114,71,tramo_3\n1,4519,113,72,tramo_3\n1,4539,112,73,tramo_3\n1,4561,111,74,tramo_3\n1,4581,110,74,tramo_3\n1,4602,108,75,tramo_3\n1,4623,107,76,tramo_3\n1,4644,106,77,tramo_3\n1,4665,105,78,tramo_3\n1,4686,104,79,tramo_3\n1,4706,103,80,tramo_3\n1,4728,102,80,tramo_3\n1,4748,101,81,tramo_3\n1,4770,100,82,tramo_3\n1,4790,99,83,tramo_3\n1,4811,98,83,tramo_3\n1,4832,97,84,tramo_3\n1,4853,96,85,tramo_3\n1,4873,95,85,tramo_3\n1,4895,95,86,tramo_3\n1,4915,94,86,tramo_3\n1,4936,93,87,tramo_3\n1,4957,93,87,tramo_3\n1,4978,92,88,tramo_3\n1,4998,92,88,tramo_3\n1,5020,91,88,tramo_3\n1,5040,91,89,tramo_3\n1,5061,90,89,tramo_3\n1,5082,90,89,tramo_3\n1,5103,90,89,tramo_3\n1,5123,90,89,tramo_3\n1,5145,90,89,tramo_3\n1,5165,90,90,tramo_3\n1,5186,90,90,pausa_3\n1,5206,90,90,pausa_3\n1,5227,90,90,pausa_3\n1,5248,90,90,pausa_3\n1,5269,90,90,pausa_3\n1,5289,90,90,pausa_3\n1,5310,90,90,pausa_3\n1,5330,90,90,pausa_3\n1,5350,90,90,pausa_3\n1,5371,90,90,pausa_3\n1,5391,90,90,pausa_3\n1,5412,90,90,pausa_3\n1,5432,90,90,pausa_3\n1,5454,90,90,pausa_3\n1,5474,90,90,pausa_3\n1,5487,90,90,pausa_3\n1,5488,90,90,fin\n'
PROCEDENCIA = {'proyecto_original': 'https://wokwi.com/projects/477503605988146177', 'fecha_captura_utc': '2026-10-25T21:58:09.733648+00:00', 'metodo': 'Ejecucion del sketch instrumentado en Wokwi, en una sesion temporal de Chrome. Lectura del monitor serie despues de presionar el pulsador. No se guardaron cambios en el proyecto publicado.', 'origen_datos': 'Consignas enteras enviadas a Servo.write(), no mediciones de posicion.', 'filas': 271, 'sha256_sketch_ino': 'a2a13cf5a223adf244429c64951b359e648a85a54877983040f4235a80ec438c', 'sha256_diagram_json': '0c012c218e0d9829fe9e388849343d6d76a1e6c7f2c7860d76161b51f20c00c4'}

In [ ]:
from IPython.display import HTML, display
import json

if MODO == "captura_incluida":
    datos = cargar_csv(CSV_INCLUIDO, ciclo=CICLO,
                       origen="Captura Wokwi · consignas de Arduino")
elif MODO == "subir_csv":
    from google.colab import files
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError("Subir un único archivo CSV o TXT por vez.")
    contenido = next(iter(subidos.values())).decode("utf-8-sig")
    datos = cargar_csv(contenido, ciclo=CICLO,
                       origen="CSV cargado · consignas de Arduino")
elif MODO == "pegar_csv":
    datos = cargar_csv(CSV_PEGADO, ciclo=CICLO,
                       origen="CSV pegado · consignas de Arduino")
elif MODO == "demostracion":
    datos = datos_demostracion()
else:
    raise ValueError("MODO no reconocido.")

resultado = informe(datos)
print(json.dumps(resultado, ensure_ascii=False, indent=2))
if not resultado["configuraciones_en_orden"] or not resultado["regresa_a_90_90"]:
    print("ATENCIÓN: este registro no contiene toda la secuencia esperada. Se reproduce lo disponible.")

## 4. Reproducir el brazo

Presionar **▶** en el reproductor. El hombro permanece en 45° durante el segundo tramo, mientras la base gira y el extremo describe un arco.

La visualización mantiene cada consigna hasta la muestra siguiente y usa los tiempos registrados. No vuelve a interpolar el CSV ni supone que las muestras están separadas exactamente 20 ms. Los dos servos conservan sus posiciones durante las pausas.

In [ ]:
fig, anim = crear_animacion(datos, fps=FPS)
guardar_html(anim, "brazo_interactivo.html", fps=FPS)
display(HTML(Path("brazo_interactivo.html").read_text(encoding="utf-8")))
plt.close(fig)

## 5. Descargar la visualización

El HTML incluye todos los cuadros y los controles y funciona sin internet. Para obtener un GIF o MP4, cambiar la opción correspondiente a `True`. El MP4 requiere FFmpeg; si no está disponible, puede usarse el HTML o el GIF. Las dimensiones de la escena son ilustrativas; el video no verifica torque, cargas ni respuesta física del servo.

In [ ]:
EXPORTAR_GIF = False
EXPORTAR_MP4 = False
DESCARGAR_ARCHIVOS = False

generados = ["brazo_interactivo.html"]
if EXPORTAR_GIF:
    anim.save("brazo.gif", writer=PillowWriter(fps=FPS), dpi=85)
    generados.append("brazo.gif")
if EXPORTAR_MP4:
    import shutil
    if not shutil.which("ffmpeg"):
        raise RuntimeError("FFmpeg no está disponible. Usar el HTML o EXPORTAR_GIF.")
    anim.save("brazo.mp4", writer=FFMpegWriter(fps=FPS, bitrate=1800), dpi=100)
    generados.append("brazo.mp4")
print("Archivos listos:", ", ".join(generados))
if DESCARGAR_ARCHIVOS:
    from google.colab import files
    for nombre in generados:
        files.download(nombre)

## 6. Obtener una nueva captura de Wokwi

1. Abrir el proyecto de Wokwi y reemplazar `sketch.ino` por el código instrumentado incluido abajo. El conexionado existente se conserva.
2. Iniciar la simulación y presionar el pulsador. Esperar hasta el comentario `# FIN_CICLO`.
3. Copiar la salida serial, incluido el encabezado `ciclo,tiempo_ms,base_deg,hombro_deg,fase`, y guardarla como `.csv` en UTF-8.
4. Volver a la sección 3 y elegir `subir_csv` o `pegar_csv`.

El registro guarda el mismo entero que se pasa a `Servo.write()`. Imprimirlo agrega una pequeña carga de procesamiento; por eso se conservan los tiempos obtenidos con `millis()`. El programa sigue teniendo tres tramos de 1,5 s y tres pausas de 0,3 s, incluida la pausa final presente en el original. El pulsador inicia la secuencia; mantenerlo presionado no la repite.

La celda siguiente prepara un ZIP con el sketch, el diagrama y la dependencia Servo. No publica ni modifica el proyecto online. Para descargarlo, cambiar `DESCARGAR_PROYECTO` a `True`.

In [ ]:
# @title Proyecto instrumentado de Arduino
SKETCH_ARDUINO = '#include <Servo.h>\n\n// Grupo 3: Mateo Flores y Federico Soler.\n// Misma secuencia del TP. Se agrega registro de las consignas por Serial.\nServo servoBase;\nServo servoHombro;\nconst int PIN_BOTON = 2;\nconst int PUNTOS[4][2] = {{90, 90}, {30, 45}, {150, 45}, {90, 90}};\nconst unsigned long T = 1500;\nconst unsigned long PASO_MS = 20;\nunsigned long inicioCiclo;\nunsigned int ciclo = 0;\nint anguloBase = 90;\nint anguloHombro = 90;\n\nvoid registrar(const char *fase) {\n  // Se registran los enteros enviados a Servo.write(), no posiciones medidas.\n  Serial.print(ciclo);\n  Serial.print(\',\');\n  Serial.print(millis() - inicioCiclo);\n  Serial.print(\',\');\n  Serial.print(anguloBase);\n  Serial.print(\',\');\n  Serial.print(anguloHombro);\n  Serial.print(\',\');\n  Serial.println(fase);\n}\n\nvoid setup() {\n  servoBase.attach(9);\n  servoHombro.attach(10);\n  pinMode(PIN_BOTON, INPUT_PULLUP);\n  servoBase.write(anguloBase);\n  servoHombro.write(anguloHombro);\n  Serial.begin(115200);\n  Serial.println("ciclo,tiempo_ms,base_deg,hombro_deg,fase");\n}\n\nvoid moverCubico(int b0, int h0, int bf, int hf,\n                 unsigned long duracion, const char *fase) {\n  unsigned long inicio = millis();\n  float s = 0;\n  while (s < 1.0) {\n    s = (millis() - inicio) / (float)duracion;\n    if (s > 1.0) s = 1.0;\n    float k = 3*s*s - 2*s*s*s;\n    // Misma conversion a int que realizaba implicitamente Servo.write().\n    anguloBase = b0 + (bf - b0) * k;\n    anguloHombro = h0 + (hf - h0) * k;\n    servoBase.write(anguloBase);\n    servoHombro.write(anguloHombro);\n    registrar(fase);\n    delay(PASO_MS);\n  }\n}\n\nvoid pausaConRegistro(unsigned long duracion, const char *fase) {\n  unsigned long inicio = millis();\n  while (millis() - inicio < duracion) {\n    registrar(fase);\n    unsigned long transcurrido = millis() - inicio;\n    if (transcurrido < duracion) {\n      unsigned long restante = duracion - transcurrido;\n      delay(restante < PASO_MS ? restante : PASO_MS);\n    }\n  }\n  registrar(fase);\n}\n\nvoid loop() {\n  if (digitalRead(PIN_BOTON) == LOW) {\n    ciclo++;\n    inicioCiclo = millis();\n    registrar("inicio");\n    const char *tramos[] = {"tramo_1", "tramo_2", "tramo_3"};\n    const char *pausas[] = {"pausa_1", "pausa_2", "pausa_3"};\n    for (int i = 0; i < 3; i++) {\n      moverCubico(PUNTOS[i][0], PUNTOS[i][1],\n                  PUNTOS[i+1][0], PUNTOS[i+1][1], T, tramos[i]);\n      // El original tambien hacia una pausa despues del tercer tramo.\n      pausaConRegistro(300, pausas[i]);\n    }\n    registrar("fin");\n    Serial.println("# FIN_CICLO");\n    // Mantener el boton presionado no repite la secuencia.\n    while (digitalRead(PIN_BOTON) == LOW) delay(10);\n  }\n}\n'
DIAGRAMA_WOKWI = '{\n  "version": 1,\n  "author": "Mateo Flores, Federico Soler",\n  "editor": "wokwi",\n  "parts": [\n    {\n      "type": "wokwi-arduino-uno",\n      "id": "uno",\n      "top": 0,\n      "left": 0,\n      "attrs": {}\n    },\n    {\n      "type": "wokwi-servo",\n      "id": "servoBase",\n      "top": -150,\n      "left": 300,\n      "attrs": {}\n    },\n    {\n      "type": "wokwi-servo",\n      "id": "servoHombro",\n      "top": -40,\n      "left": 300,\n      "attrs": {}\n    },\n    {\n      "type": "wokwi-pushbutton",\n      "id": "btn",\n      "top": 227,\n      "left": 297.6,\n      "attrs": {\n        "color": "green"\n      }\n    }\n  ],\n  "connections": [\n    [\n      "servoBase:PWM",\n      "uno:9",\n      "orange",\n      []\n    ],\n    [\n      "servoBase:V+",\n      "uno:5V",\n      "red",\n      []\n    ],\n    [\n      "servoBase:GND",\n      "uno:GND.1",\n      "black",\n      []\n    ],\n    [\n      "servoHombro:PWM",\n      "uno:10",\n      "orange",\n      []\n    ],\n    [\n      "servoHombro:V+",\n      "uno:5V",\n      "red",\n      []\n    ],\n    [\n      "servoHombro:GND",\n      "uno:GND.1",\n      "black",\n      []\n    ],\n    [\n      "btn:1.l",\n      "uno:2",\n      "green",\n      []\n    ],\n    [\n      "btn:2.l",\n      "uno:GND.2",\n      "black",\n      []\n    ]\n  ],\n  "dependencies": {}\n}'
DESCARGAR_PROYECTO = False

import zipfile
with zipfile.ZipFile("Wokwi_Grupo3_con_registro.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("sketch.ino", SKETCH_ARDUINO)
    z.writestr("diagram.json", DIAGRAMA_WOKWI)
    z.writestr("libraries.txt", "Servo\n")
    z.writestr("captura_wokwi.csv", CSV_INCLUIDO)
    z.writestr("procedencia.json", json.dumps(PROCEDENCIA, ensure_ascii=False, indent=2))
print("Creado: Wokwi_Grupo3_con_registro.zip")
if DESCARGAR_PROYECTO:
    from google.colab import files
    files.download("Wokwi_Grupo3_con_registro.zip")

## Alcance y referencias

La animación es una **visualización cinemática de las consignas angulares**. No representa mediciones de encoder, realimentación del programa Arduino, cargas, torque ni colisiones. La prueba de las posiciones se refiere a las órdenes enviadas y la geometría elegida.

- Universidad de Palermo. *Trayectorias* e *Implementación de las trayectorias robóticas*, apuntes del módulo 6.
- [Proyecto original en Wokwi](https://wokwi.com/projects/477503605988146177).
- [Documentación del monitor serie](https://docs.wokwi.com/guides/serial-monitor).
- [Servo de Wokwi](https://docs.wokwi.com/parts/wokwi-servo).
- [Animaciones con Matplotlib](https://matplotlib.org/stable/api/animation_api.html).
- [Gráficos 3D con Matplotlib](https://matplotlib.org/stable/gallery/mplot3d/index.html).